# 01 – Data Acquisition and Preprocessing

## Urban Heat Island Analysis – Voi Town, Kenya

This notebook covers the definition of the study area, Landsat 8 and 9 data acquisition, image filtering, cloud masking and preprocessing using Google Earth Engine.

In [56]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


In [57]:
ee.Authenticate()

ee.Initialize(
    project='voi-urban-heat-island-analysis'
)

print("Google Earth Engine connected successfully!")

Google Earth Engine connected successfully!


In [58]:
# Approximate Voi town centre
voi_point = ee.Geometry.Point([38.56, -3.39])

# Create a 5 km buffer
voi_aoi = voi_point.buffer(5000)

print("Study area type:", voi_aoi.getInfo()['type'])

Study area type: Polygon


In [59]:
Map = geemap.Map()

Map.centerObject(voi_aoi, 12)

Map.addLayer(
    voi_aoi,
    {'color': 'red'},
    'Voi Study Area'
)

Map

Map(center=[-3.3899948705567735, 38.56000014249622], controls=(WidgetControl(options=['position', 'transparent…

In [60]:
# Landsat 8 Collection 2 Level-2
landsat_8 = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(voi_aoi)
    .filterDate('2016-01-01', '2026-10-01')
)

# Landsat 9 Collection 2 Level-2
landsat_9 = (
    ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
    .filterBounds(voi_aoi)
    .filterDate('2022-01-01', '2026-10-01')
)

# Merge both collections
landsat = landsat_8.merge(landsat_9)

print("Landsat 8 images:", landsat_8.size().getInfo())
print("Landsat 9 images:", landsat_9.size().getInfo())
print("Total images:", landsat.size().getInfo())

Landsat 8 images: 231
Landsat 9 images: 100
Total images: 331


In [61]:
# Keep only Landsat scenes with surface temperature data
landsat = landsat.filter(
    ee.Filter.eq('PROCESSING_LEVEL', 'L2SP')
)

print("L2SP scenes:", landsat.size().getInfo())

L2SP scenes: 331


In [62]:
def preprocess_landsat(image):

    # Cloud and shadow masking
    qa = image.select('QA_PIXEL')

    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
        .And(qa.bitwiseAnd(1 << 5).eq(0))
    )

    # Remove saturated pixels
    mask = mask.And(image.select('QA_RADSAT').eq(0))

    # Scale optical bands
    optical = image.select([
        'SR_B2', 'SR_B3', 'SR_B4',
        'SR_B5', 'SR_B6', 'SR_B7'
    ]).multiply(0.0000275).add(-0.2)

    # Calculate NDVI
    ndvi = optical.normalizedDifference(
        ['SR_B5', 'SR_B4']
    ).rename('NDVI')

    # Convert surface temperature to Celsius
    lst = (
        image.select('ST_B10')
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename('LST_C')
    )

    # Combine processed bands
    processed = (
        image.addBands(optical, overwrite=True)
        .addBands(ndvi)
        .addBands(lst)
    )

    return (
        processed.updateMask(mask)
        .copyProperties(image, ['system:time_start'])
    )

print("Preprocessing function created successfully.")

Preprocessing function created successfully.


In [63]:
# Apply preprocessing to Landsat collection
landsat_processed = landsat.map(preprocess_landsat)

print(
    "Processed Landsat scenes:",
    landsat_processed.size().getInfo()
)

Processed Landsat scenes: 331


In [64]:
# Create June–September seasonal composites
years = list(range(2016, 2027))

annual_composites = {}

for year in years:

    start_date = f'{year}-06-01'
    end_date = f'{year}-10-01'

    composite = (
        landsat_processed
        .filterDate(start_date, end_date)
        .median()
        .clip(voi_aoi)
    )

    annual_composites[year] = composite

print("Seasonal composites created:", len(annual_composites))
print("Available years:", list(annual_composites.keys()))

Seasonal composites created: 11
Available years: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [65]:
# Check the number of images contributing to each seasonal composite
for year in years:
    count = (
        landsat_processed
        .filterDate(f'{year}-06-01', f'{year}-10-01')
        .size()
        .getInfo()
    )

    print(f"{year}: {count} scenes")

2016: 8 scenes
2017: 7 scenes
2018: 6 scenes
2019: 8 scenes
2020: 8 scenes
2021: 7 scenes
2022: 16 scenes
2023: 15 scenes
2024: 12 scenes
2025: 15 scenes
2026: 12 scenes


In [66]:
# Select 2025 composite
image_2025 = annual_composites[2025]

# Display LST
Map = geemap.Map()
Map.centerObject(voi_aoi, 12)

Map.addLayer(
    image_2025.select('LST_C'),
    {
        'min': 25,
        'max': 50,
        'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']
    },
    '2025 Land Surface Temperature'
)

Map.addLayer(
    voi_aoi,
    {'color': 'white'},
    'Voi Study Area'
)

Map

Map(center=[-3.3899948705567735, 38.56000014249622], controls=(WidgetControl(options=['position', 'transparent…